# Voice Clone - Chatterbox & RNNoise Backend for Google Colab

This notebook allows you to run the FastAPI backend for Chatterbox and RNNoise on a Google Colab GPU instance and tunnel it to your local frontend.

**Important**: Ensure you are using a GPU runtime! Go to **Runtime** > **Change runtime type** and select **T4 GPU** (or any other available GPU).

In [1]:
# 1. Check GPU availability
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available. Inference will be extremely slow. Please switch to GPU runtime!")

GPU available: True
Device name: Tesla T4


## Install Dependencies & Compile RNNoise

We will install the official **Chatterbox** library (by Resemble AI) along with PyTorch 2.6.0 / torchaudio 2.6.0 and build `rnnoise` from source. Running this cell will automatically restart the Colab kernel to load the new packages.

In [ ]:
# 2. Clone and install Chatterbox
!git clone https://github.com/resemble-ai/chatterbox.git
!pip install -e ./chatterbox

# 3. Upgrade pip tools and clean up PyTorch versions
!pip install -q --upgrade pip setuptools wheel
!pip uninstall -y torch torchvision torchaudio
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0

# 4. Clone and build RNNoise (C-based noise suppression)
!git clone https://github.com/xiph/rnnoise.git
!apt-get install -y autoconf libtool
!cd rnnoise && ./autogen.sh && ./configure && make

# 5. Install FastAPI and other server/dependency packages
!pip install fastapi uvicorn python-multipart requests

# 6. Restart Colab kernel to load the new environment
import os
print("Restarting kernel to load new library versions...")
os.kill(os.getpid(), 9)

Cloning into 'chatterbox'...
remote: Enumerating objects: 328, done.
remote: Total 328 (delta 0), reused 0 (delta 0), pack-reused 328 (from 1)
Receiving objects: 100% (328/328), 1.51 MiB | 3.77 MiB/s, done.
Resolving deltas: 100% (119/119), done.
Obtaining file:///content/chatterbox
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Cloning https://github.com/resemble-ai/Perth.git (to revision master) to /tmp/pip-install-zg5wj1t7/resemble-perth_1ae67875300f45a4b39ca8957bf545f5
  Running command git clone --filter=blob:none --quiet https://github.com/resemble-ai/Perth.git /tmp/pip-install-zg5wj1t7/resemble-perth_1ae67875300f45a4b39ca8957bf545f5
  Resolved https://github.com/resemble-ai/Perth.git to commit ce86c49d029f42272c1902eccb675556b9ed2330
  Installing build dependencies ... done
  Getting requirements to build wheel ... don

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torchvision 0.26.0+cu128 requires torch==2.11.0, but you have torch 2.6.0 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
Found existing installation: torch 2.6.0
Uninstalling torch-2.6.0:
  Successfully uninstalled torch-2.6.0
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.6.0
Uninstalling torchaudio-2.6.0:
  Successfully uninstalled torchaudio-2.6.0


y
Cloning into 'rnnoise'...
remote: Enumerating objects: 907, done.
remote: Counting objects: 100% (426/426), done.
remote: Compres

## Create the Backend Application File (`main.py`)

Running the cell below will write the FastAPI backend code directly into a local file named `main.py` on your Colab instance.

In [1]:
%%writefile main.py
import os
import sys
import uuid
import shutil
import base64
import traceback
import subprocess
import torch
import requests
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Response
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse

app = FastAPI(
    title="Chatterbox & RNNoise API",
    description="FastAPI backend for Chatterbox Multilingual & Turbo TTS and RNNoise voice suppression"
)

# CORS middleware configuration to communicate with Vite React Frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Detect the best available computing device
if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"--- Voice Clone Backend starting on device: {DEVICE} ---")

# Setup temporary directory for uploads and generations
base_dir = os.path.dirname(os.path.abspath(__file__))
TEMP_DIR = os.path.join(base_dir, "temp")
os.makedirs(TEMP_DIR, exist_ok=True)

# ==============================================================
# In-memory registry for uploaded reference audio files.
# Maps reference_id (str) -> absolute file path (str).
# Files persist for the lifetime of the backend/Colab session.
# ==============================================================
reference_store: dict[str, str] = {}

# Lazy loading of models to optimize memory and startup speed
multilingual_model = None
turbo_model = None

def get_multilingual_model():
    global multilingual_model
    if multilingual_model is None:
        print("[PROGRESS] Loading Chatterbox Multilingual TTS (v3)...")
        from chatterbox.mtl_tts import ChatterboxMultilingualTTS
        multilingual_model = ChatterboxMultilingualTTS.from_pretrained(
            device=DEVICE,
            t3_model="v3"
        )
        print("[PROGRESS] Chatterbox Multilingual TTS loaded successfully.")
    return multilingual_model

def get_turbo_model():
    global turbo_model
    if turbo_model is None:
        print("[PROGRESS] Loading Chatterbox Turbo TTS...")
        from chatterbox.tts_turbo import ChatterboxTurboTTS
        turbo_model = ChatterboxTurboTTS.from_pretrained(
            device=DEVICE
        )
        print("[PROGRESS] Chatterbox Turbo TTS loaded successfully.")
    return turbo_model

# Map frontend language names to Chatterbox language codes
LANG_MAP = {
    "english": "en",
    "spanish": "es",
    "french": "fr",
    "german": "de",
    "hindi": "hi",
    "chinese": "zh",
    "japanese": "ja",
    "italian": "it",
    "portuguese": "pt",
}

def get_language_id(lang_name: str) -> str:
    cleaned = lang_name.lower().strip()
    for k, v in LANG_MAP.items():
        if k in cleaned:
            return v
    return "en" # Fallback to English

def denoise_with_rnnoise(input_path: str, temp_dir: str) -> str:
    raw_in = os.path.join(temp_dir, f"in_{uuid.uuid4()}.raw")
    raw_out = os.path.join(temp_dir, f"out_{uuid.uuid4()}.raw")
    wav_out = os.path.join(temp_dir, f"denoised_{uuid.uuid4()}.wav")

    # Locate rnnoise_demo binary
    rnnoise_bin = None
    possible_paths = [
        "/content/rnnoise/examples/rnnoise_demo",
        "rnnoise/examples/rnnoise_demo",
        "./rnnoise/examples/rnnoise_demo",
        "../rnnoise/examples/rnnoise_demo",
        os.path.join(os.path.dirname(temp_dir), "rnnoise", "examples", "rnnoise_demo")
    ]
    for p in possible_paths:
        if os.path.exists(p):
            rnnoise_bin = p
            break
    if rnnoise_bin is None:
        rnnoise_bin = shutil.which("rnnoise_demo")
        if rnnoise_bin is None:
            raise HTTPException(
                status_code=500,
                detail="rnnoise_demo binary not found. Please compile rnnoise in Colab."
            )

    try:
        # Step 1: Convert input audio to raw 16-bit PCM 48kHz mono
        print(f"[PROGRESS] Converting input to raw PCM: {input_path}")
        cmd1 = ["ffmpeg", "-y", "-i", input_path, "-f", "s16le", "-ac", "1", "-ar", "48000", raw_in]
        subprocess.run(cmd1, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        # Step 2: Run rnnoise_demo
        print(f"[PROGRESS] Running RNNoise demo suppression...")
        cmd2 = [rnnoise_bin, raw_in, raw_out]
        subprocess.run(cmd2, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        # Step 3: Convert raw output back to WAV
        print(f"[PROGRESS] Converting raw output back to WAV: {wav_out}")
        cmd3 = ["ffmpeg", "-y", "-f", "s16le", "-ac", "1", "-ar", "48000", "-i", raw_out, wav_out]
        subprocess.run(cmd3, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        return wav_out
    except subprocess.CalledProcessError as e:
        error_msg = e.stderr.decode() if e.stderr else str(e)
        print(f"Error during rnnoise subprocess call: {error_msg}")
        raise HTTPException(status_code=500, detail=f"Denoising failed: {error_msg}")
    finally:
        # Cleanup intermediate raw files
        if os.path.exists(raw_in):
            os.remove(raw_in)
        if os.path.exists(raw_out):
            os.remove(raw_out)

@app.get("/")
async def root():
    return {"message": "Chatterbox & RNNoise API is running!"}

# ==============================================================
# NEW ENDPOINT: Upload reference audio once, get a reusable ID
# ==============================================================
@app.post("/api/upload_reference")
async def upload_reference(
    audio_prompt: UploadFile = File(...)
):
    """
    Accepts a reference audio file, stores it on disk, and returns
    a reference_id that can be used in subsequent /api/tts calls
    instead of re-uploading the audio every time.
    """
    try:
        ref_id = str(uuid.uuid4())
        ext = os.path.splitext(audio_prompt.filename)[1] or ".wav"
        stored_filename = f"ref_{ref_id}{ext}"
        stored_path = os.path.join(TEMP_DIR, stored_filename)

        with open(stored_path, "wb") as buffer:
            shutil.copyfileobj(audio_prompt.file, buffer)

        # Register in the in-memory store
        reference_store[ref_id] = stored_path

        print(f"[REFERENCE] Stored reference audio: {ref_id} -> {stored_filename}")
        return JSONResponse(content={"reference_id": ref_id})

    except Exception as e:
        print(f"Error uploading reference audio: {e}")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"Failed to store reference audio: {str(e)}")

# ==============================================================
# UPDATED ENDPOINT: TTS with optional reference_id support
# ==============================================================
@app.post("/api/tts")
async def text_to_speech(
    text: str = Form(...),
    exaggeration: float = Form(0.5),
    cfg_weight: float = Form(0.5),
    temperature: float = Form(0.7),
    seed: int = Form(0),
    min_p: float = Form(0.05),
    top_p: float = Form(0.7),
    repetition_penalty: float = Form(1.2),
    language: str = Form("English"),
    model: str = Form("Chatterbox v2.1-Turbofast"),
    audio_prompt: UploadFile = File(None),
    reference_id: str = Form(None)
):
    prompt_path = None
    delete_prompt = False
    try:
        # ---- Determine the reference audio source ----
        has_audio_prompt = audio_prompt is not None and audio_prompt.filename
        has_reference_id = reference_id is not None and reference_id.strip() != ""

        # Invalid: both provided simultaneously
        if has_audio_prompt and has_reference_id:
            raise HTTPException(
                status_code=400,
                detail="Cannot provide both audio_prompt and reference_id. Use one or the other."
            )

        if has_reference_id:
            # Case 1: Use a previously uploaded reference by ID
            ref_id_clean = reference_id.strip()
            stored_path = reference_store.get(ref_id_clean)
            if stored_path is None:
                raise HTTPException(
                    status_code=404,
                    detail=f"Reference ID not found. It may have expired if the backend restarted."
                )
            if not os.path.exists(stored_path):
                # File was in registry but missing from disk
                del reference_store[ref_id_clean]
                raise HTTPException(
                    status_code=404,
                    detail=f"Reference audio file no longer exists on disk."
                )
            prompt_path = stored_path
            delete_prompt = False  # Never delete reference store files
            print(f"[TTS] Using stored reference: {ref_id_clean}")

        elif has_audio_prompt:
            # Case 2: Direct file upload (existing behavior)
            ext = os.path.splitext(audio_prompt.filename)[1] or ".wav"
            prompt_path = os.path.join(TEMP_DIR, f"prompt_{uuid.uuid4()}{ext}")
            with open(prompt_path, "wb") as buffer:
                shutil.copyfileobj(audio_prompt.file, buffer)
            delete_prompt = True

        else:
            # Case 3: No reference provided — use default voice
            prompt_path = os.path.join(TEMP_DIR, "default_voice.wav")
            if not os.path.exists(prompt_path):
                print("[PROGRESS] Downloading default reference voice...")
                default_url = "https://raw.githubusercontent.com/resemble-ai/chatterbox/main/samples/input.wav"
                r = requests.get(default_url)
                r.raise_for_status()
                with open(prompt_path, "wb") as f:
                    f.write(r.content)
                print("[PROGRESS] Default reference voice downloaded successfully.")

        # Determine which Chatterbox model to use
        is_turbo = "turbo" in model.lower()
        import torchaudio as ta

        if is_turbo:
            print(f"[PROGRESS] Synthesizing speech with Chatterbox Turbo TTS. Text: '{text[:50]}...'")
            tts_model = get_turbo_model()
            wav = tts_model.generate(
                text=text,
                audio_prompt_path=prompt_path
            )
        else:
            lang_id = get_language_id(language)
            print(f"[PROGRESS] Synthesizing speech with Chatterbox Multilingual TTS. Language: {lang_id}, Text: '{text[:50]}...'")
            tts_model = get_multilingual_model()
            wav = tts_model.generate(
                text=text,
                language_id=lang_id,
                audio_prompt_path=prompt_path
            )

        out_filename = f"tts_{uuid.uuid4()}.wav"
        out_path = os.path.join(TEMP_DIR, out_filename)

        # Save output wav file
        ta.save(out_path, wav.cpu(), tts_model.sr)

        if delete_prompt and prompt_path and os.path.exists(prompt_path):
            os.remove(prompt_path)

        return FileResponse(out_path, media_type="audio/wav", filename="tts_output.wav")

    except HTTPException:
        raise  # Re-raise HTTP exceptions as-is
    except Exception as e:
        print(f"Error during TTS generation: {e}")
        traceback.print_exc()
        if delete_prompt and prompt_path and os.path.exists(prompt_path):
            os.remove(prompt_path)
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/denoise")
async def denoise_audio(
    audio: UploadFile = File(...)
):
    input_path = None
    try:
        ext = os.path.splitext(audio.filename)[1] or ".wav"
        input_path = os.path.join(TEMP_DIR, f"denoise_in_{uuid.uuid4()}{ext}")
        with open(input_path, "wb") as buffer:
            shutil.copyfileobj(audio.file, buffer)

        denoised_path = denoise_with_rnnoise(input_path, TEMP_DIR)

        if input_path and os.path.exists(input_path):
            os.remove(input_path)

        return FileResponse(denoised_path, media_type="audio/wav", filename="denoised_output.wav")
    except HTTPException:
        raise
    except Exception as e:
        print(f"Error during denoising: {e}")
        traceback.print_exc()
        if input_path and os.path.exists(input_path):
            os.remove(input_path)
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/api/health")
async def health():
    chatterbox_online = False
    try:
        import chatterbox
        chatterbox_online = True
    except ImportError:
        pass

    return {
        "status": "online",
        "device": DEVICE,
        "tts_loaded": chatterbox_online,
        "vc_loaded": chatterbox_online
    }

if __name__ == "__main__":
    import uvicorn
    uvicorn.run("main:app", host="127.0.0.1", port=8000, reload=True)


Writing main.py


## Run Cloudflare Tunnel

We will download and run Cloudflare Tunnel (`cloudflared`) to expose the backend on a public URL. Copy the printed URL (ending with `.trycloudflare.com`) and paste it in your React frontend settings.

In [2]:
# 7. Download and install cloudflared
import urllib.request
import os

if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Downloading cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb
    print("cloudflared installed successfully!")
else:
    print("cloudflared already installed.")

Selecting previously unselected package cloudflared.
(Reading database ... 122512 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.7.3) ...
Setting up cloudflared (2026.7.3) ...
Processing triggers for man-db (2.10.2-1) ...
cloudflared installed successfully!


In [1]:
# 8. Start cloudflared tunnel in background thread and show Copy Button
import subprocess
import threading
import time
import re
from google.colab import output
import ipywidgets as widgets
from IPython.display import display

# Global variable to store the detected URL
PUBLIC_URL = ""

def run_tunnel():
    global PUBLIC_URL
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "trycloudflare.com" in line:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                PUBLIC_URL = match.group(0)
                print("\n" + "="*80)
                print(f"  YOUR PUBLIC BACKEND URL (Copy this URL into the frontend Settings Page):")
                print(f"  {PUBLIC_URL}")
                print("="*80 + "\n")

def copy_url_btn(b):
    if PUBLIC_URL:
        output.eval_js(f"navigator.clipboard.writeText('{PUBLIC_URL}')")
        print(f"Copied to clipboard: {PUBLIC_URL}")
    else:
        print("URL not ready yet. Please wait a moment...")

# Start background tunnel
threading.Thread(target=run_tunnel, daemon=True).start()

# Display the Copy Button
button = widgets.Button(description="Copy Backend URL", button_style='success', icon='copy')
button.on_click(copy_url_btn)
display(button)

# Wait a few seconds to let the tunnel initialize
time.sleep(5)

Button(button_style='success', description='Copy Backend URL', icon='copy', style=ButtonStyle())

## Start the Backend FastAPI Server

Now, start the backend server. It will load the selected Chatterbox models dynamically upon the first request and run on port 8000.

In [ ]:
# 9. Start FastAPI server using uvicorn on port 8000
print("Starting FastAPI server...")
!uvicorn main:app --host 0.0.0.0 --port 8000

[PROGRESS] Downloading default reference voice...
Error during TTS generation: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/resemble-ai/chatterbox/main/samples/input.wav
Traceback (most recent call last):
  File "/content/main.py", line 177, in text_to_speech
    r.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/resemble-ai/chatterbox/main/samples/input.wav
INFO:     2401:9620:205:2d5:285b:6b7:2c27:ad11:0 - "POST /api/tts HTTP/1.1" 500 Internal Server Error
[PROGRESS] Downloading default reference voice...
Error during TTS generation: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/resemble-ai/chatterbox/main/samples/input.wav
Traceback (most recent call last):
  File "/content/main.py", line 177, in text_to_speech
  